In [1]:
import os
from dotenv import load_dotenv
load_dotenv() 

True

In [2]:
from openai import OpenAI
import os

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

try:
    response = client.models.list()
    print("Success!")
except Exception as e:
    print(e)

Success!


In [3]:
from langchain_openai  import ChatOpenAI
llm=chat=ChatOpenAI(model="gpt-5-nano", temperature=0)
llm

ChatOpenAI(metadata={'lc_versions': {'langchain-core': '1.4.7', 'langchain-openai': '1.3.2'}}, output_version=None, profile={'name': 'GPT-5 Nano', 'release_date': '2025-08-07', 'last_updated': '2025-08-07', 'open_weights': False, 'max_input_tokens': 272000, 'max_output_tokens': 128000, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': False, 'image_url_inputs': True, 'pdf_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_choice': True, 'tool_call_streaming': True}, client=<openai.resources.chat.completions.completions.Completions object at 0x10fd4fe90>, async_client=<openai.resources.chat.completions.completions.AsyncCompletions object at 0x110140980>, root_client=<openai.OpenAI object at 0x10fd4ec90>, root_async_client=<o

# *Rag impementation with pdf and my own data

# *step 1: Extracting the data from pdf

In [4]:
from langchain_community.document_loaders import PyPDFLoader
print(dir(PyPDFLoader))

['__abstractmethods__', '__class__', '__del__', '__delattr__', '__dict__', '__dir__', '__doc__', '__eq__', '__format__', '__ge__', '__getattribute__', '__getstate__', '__gt__', '__hash__', '__init__', '__init_subclass__', '__le__', '__lt__', '__module__', '__ne__', '__new__', '__reduce__', '__reduce_ex__', '__repr__', '__setattr__', '__sizeof__', '__slots__', '__str__', '__subclasshook__', '__weakref__', '_abc_impl', '_is_s3_presigned_url', '_is_s3_url', '_is_valid_url', 'alazy_load', 'aload', 'lazy_load', 'load', 'load_and_split', 'source']


/var/folders/2z/pqpq9wcs0dv4jfbgkp96j3780000gn/T/ipykernel_10447/3345968884.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


In [5]:
pdf_path="/Users/bilal/Downloads/RAG/Doc_pdf/fabric-admin.pdf"
pdf_path

'/Users/bilal/Downloads/RAG/Doc_pdf/fabric-admin.pdf'

In [6]:
loader=PyPDFLoader(pdf_path)
docs=loader.load()
docs

[Document(metadata={'producer': 'Microsoft Learn PDF 1.0.25309.01', 'creator': 'Microsoft Learn', 'creationdate': '2025-12-09T19:42:08+00:00', 'title': 'fabric admin | Microsoft Learn', 'moddate': '2025-12-09T19:42:08+00:00', 'source': '/Users/bilal/Downloads/RAG/Doc_pdf/fabric-admin.pdf', 'total_pages': 290, 'page': 0, 'page_label': '1'}, page_content='Tell us about your PDF experience.\nMicrosoft Fabric documentation for\nadmins\nLearn about the Microsoft Fabric admin settings, options, and tools.\nFabric in your organization\nｅOVERVIEW\nWhat is Microsoft Fabric admin?\nWhat is the admin portal?\nｂGET STARTED\nEnable Fabric for your organization\nRegion availability\nFind your Fabric home region\nｃHOW-TO GUIDE\nUnderstand Fabric admin roles\nｉREFERENCE\nGovernance documentation\nSecurity documentation\nTools and settings\nｅOVERVIEW\nAbout tenant settings\nｃHOW-TO GUIDE\nSet up git integration\nSet up item certification'),
 Document(metadata={'producer': 'Microsoft Learn PDF 1.0.25309

In [ ]:
#change metadata of the document

for i in docs:
    i.metadata={
        "source":"MFL.PDF",
        "develped_by":"Bilal"

    }

In [8]:
docs[0].metadata

{'source': 'MFL.PDF', 'develped_by': 'Bilal'}

# *step 2 : split the document into chunks and create vector store*

In [9]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter=RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=100)


In [10]:
chunks=splitter.split_documents(docs)
chunks

[Document(metadata={'source': 'MFL.PDF', 'develped_by': 'Bilal'}, page_content='Tell us about your PDF experience.\nMicrosoft Fabric documentation for\nadmins\nLearn about the Microsoft Fabric admin settings, options, and tools.\nFabric in your organization\nｅOVERVIEW\nWhat is Microsoft Fabric admin?\nWhat is the admin portal?\nｂGET STARTED\nEnable Fabric for your organization\nRegion availability\nFind your Fabric home region\nｃHOW-TO GUIDE\nUnderstand Fabric admin roles\nｉREFERENCE\nGovernance documentation\nSecurity documentation\nTools and settings\nｅOVERVIEW\nAbout tenant settings\nｃHOW-TO GUIDE\nSet up git integration\nSet up item certification'),
 Document(metadata={'source': 'MFL.PDF', 'develped_by': 'Bilal'}, page_content='Configure notifications\nSet up metadata scanning\nEnable content certification\nEnable service principal authentication\nConfigure Multi-Geo support\nMonitoring and management\nｅOVERVIEW\nWhat is the admin monitoring workspace?\nｐCONCEPT\nFeature usage and 

In [ ]:
len(chunks) # we have 522 embeding vectors for our pdf document

522

# *step 3 : creating embedding for the chunks and storing them in vector database (we will use chroma db for this)*

In [12]:
from langchain_openai import OpenAIEmbeddings
openai_embeddings=OpenAIEmbeddings(model="text-embedding-3-small")

openai_embeddings



OpenAIEmbeddings(client=<openai.resources.embeddings.Embeddings object at 0x11138a060>, async_client=<openai.resources.embeddings.AsyncEmbeddings object at 0x1115c59d0>, model='text-embedding-3-small', dimensions=None, deployment='text-embedding-ada-002', openai_api_version=None, openai_api_base=None, openai_api_type=None, openai_proxy=None, embedding_ctx_length=8191, openai_api_key=SecretStr('**********'), openai_organization=None, allowed_special=None, disallowed_special=None, chunk_size=1000, max_retries=2, request_timeout=None, headers=None, tiktoken_enabled=True, tiktoken_model_name=None, show_progress_bar=False, model_kwargs={}, skip_empty=False, default_headers=None, default_query=None, retry_min_seconds=4, retry_max_seconds=20, http_client=None, http_async_client=None, check_embedding_ctx_length=True)

# *step 4 : store the embeddings in the exiting and local vector database*

In [17]:
from langchain_community.vectorstores import Chroma

#vectordb=Chroma.from_documents(documents=chunks, 
#                               embedding=openai_embeddings,
#                               persist_directory="/Users/bilal/Downloads/RAG/vectorDB")
#
#vectordb

vectordb=Chroma(persist_directory="/Users/bilal/Downloads/RAG/vectorDB", embedding_function=openai_embeddings)

In [18]:
#now add the vector in data base
vectordb.add_documents(chunks)

['e11c268c-faf9-4e9a-a6ed-95222742b146',
 'eb780712-ec13-45db-a570-b4303f4e1aa1',
 '9679ad81-958a-4949-b67f-f3d8aa974851',
 '9bf99f9e-c47c-4965-82e9-cb1d5d6257aa',
 '0c5e0c84-91d8-48a6-a664-a40167e3447e',
 '71c26cb6-54a4-458d-a76b-5a9a273cc715',
 'b46655f2-e9e2-4f43-96d9-fea3edf95f50',
 '11fa66bc-4905-4714-8e09-9157b6f15d6d',
 '3a37906a-7929-4769-9d69-c6d3b914a745',
 '92c52ef2-fafb-4c48-b199-8cabeb76d082',
 'fd74558e-da36-4cb7-9e9d-2b26e399d09a',
 '0380d370-9615-4d6b-b985-20b7d7109fd1',
 '46eaa6ad-d43b-4e42-8b2a-5ec3d8d615c2',
 '0622e943-33a3-441d-afb9-009f348fa0b5',
 '20fdc059-a8b4-453b-bb3f-f8a74223e084',
 'e62b0f02-5e4e-46da-8946-c2e5cf873a25',
 'd9f089b7-a463-4de7-8e12-3def84377564',
 '5940b401-2a32-4dc9-9ff9-c9236b60825c',
 '5adb4720-bd15-4745-8958-3cd6f1558934',
 '11fa554a-c8e7-4ab4-b57b-15c876b4a19c',
 'e7eca06b-3691-448a-a9f7-5253a33d24f2',
 'b3222a37-9d83-4d45-b486-65e1e8c83945',
 'bc45402d-ec42-431a-9637-484976b6600e',
 '36a292a6-1943-4238-b7bf-9e3d8d30ee32',
 '0451f1cb-8cd0-

# *step 5 : Semantic search*

In [ ]:
#ask the question to the vector data base
vectordb.similarity_search("What is microsoft fabric admin?", k=3)

[Document(metadata={'develped_by': 'Bilal', 'source': 'MFL.PDF'}, page_content='What is Microsoft Fabric admin?\n07/01/2024\nMicrosoft Fabric admin is the management of the organization-wide settings that control how\nMicrosoft Fabric works. Users that are assigned to admin roles configure, monitor, and\nprovision organizational resources. This article provides an overview of admin roles, tasks, and\ntools to help you get started.\nThere are several roles that work together to administer Microsoft Fabric for your organization.\nMost admin roles are assigned in the Microsoft 365 admin portal or by using PowerShell. The\ncapacity admin roles are assigned when the capacity is created. To learn more about each of\nthe admin roles, see About admin roles. To learn how to assign admin roles, see Assign admin\nroles.\nThis section lists the Microsoft 365 admin roles and the tasks they can perform.\nGlobal administrator\nUnlimited access to all management features for the organization\nAssign r

# *Resuse the vectore database*

In [ ]:

from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAIEmbeddings

embeddings_model_2=OpenAIEmbeddings(model="text-embedding-3-small")


vectordb_persisted=Chroma(persist_directory="/Users/bilal/Downloads/RAG/vectorDB", embedding_function=embeddings_model_2)

In [ ]:
vectordb_persisted.similarity_search("What is fabric onelake?", k=3)